In [ ]:
!pip install  opencv-python tqdm requests
!pip install mediapipe==0.10.21

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.2/81.2 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 10.8 MB/s eta 0:00:00
  

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
import requests

url = "https://raw.githubusercontent.com/dxli94/WLASL/master/start_kit/WLASL_v0.3.json"

response = requests.get(url)
data = response.json()

selected_glosses = [item['gloss'] for item in data if len(item['instances']) >= 21]

print("Total classes:", len(selected_glosses))

Total classes: 32


In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/sanket2"

RAW_DATA = BASE_PATH + "/raw_videos"
PROCESSED_DATA = BASE_PATH + "/processed_data"

os.makedirs(RAW_DATA, exist_ok=True)
os.makedirs(PROCESSED_DATA, exist_ok=True)

print("Folders created")

Folders created


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import os
import json
import requests
from tqdm import tqdm

In [ ]:
VIDEOS_PER_CLASS = 21

def download_video(url, path):
    try:
        r = requests.get(url)
        with open(path, "wb") as f:
            f.write(r.content)
    except:
        pass

In [ ]:
for item in data:

    word = item["gloss"]

    if word not in selected_glosses:
        continue

    word_folder = f"{RAW_DATA}/{word}"
    os.makedirs(word_folder, exist_ok=True)

    count = 0

    for instance in item["instances"]:

        if count >= VIDEOS_PER_CLASS:
            break

        video_url = instance["url"]
        video_id = instance["video_id"]

        save_path = f"{word_folder}/{video_id}.mp4"

        download_video(video_url, save_path)

        count += 1

In [ ]:
selected_glosses

['book',
 'drink',
 'computer',
 'before',
 'chair',
 'go',
 'clothes',
 'who',
 'candy',
 'cousin',
 'deaf',
 'fine',
 'help',
 'no',
 'thin',
 'walk',
 'year',
 'yes',
 'all',
 'black',
 'cool',
 'finish',
 'hot',
 'like',
 'many',
 'mother',
 'now',
 'orange',
 'table',
 'thanksgiving',
 'what',
 'woman']

In [ ]:
gloss_video_counts = {}

for gloss in selected_glosses:
    word_folder = os.path.join(RAW_DATA, gloss)
    if os.path.exists(word_folder):
        gloss_video_counts[gloss] = len(os.listdir(word_folder))
    else:
        gloss_video_counts[gloss] = 0

print("Video counts per gloss:")
for gloss, count in gloss_video_counts.items():
    print(f"{gloss}: {count}")


Video counts per gloss:
book: 8
drink: 18
computer: 20
before: 19
chair: 15
go: 19
clothes: 12
who: 17
candy: 17
cousin: 19
deaf: 19
fine: 18
help: 17
no: 19
thin: 18
walk: 18
year: 18
yes: 19
all: 19
black: 19
cool: 19
finish: 19
hot: 20
like: 20
many: 20
mother: 18
now: 20
orange: 19
table: 13
thanksgiving: 20
what: 19
woman: 18


In [ ]:
import os

data_dir = "/content/drive/MyDrive/sanket2/raw_videos"

classes = sorted(os.listdir(data_dir))
print("Classes:", classes)
print("Total classes:", len(classes))

# Label mapping
gloss_to_id = {cls:i for i, cls in enumerate(classes)}
id_to_gloss = {i:cls for cls,i in gloss_to_id.items()}

Classes: ['all', 'before', 'black', 'book', 'candy', 'chair', 'clothes', 'computer', 'cool', 'cousin', 'deaf', 'drink', 'fine', 'finish', 'go', 'help', 'hot', 'like', 'many', 'mother', 'no', 'now', 'orange', 'table', 'thanksgiving', 'thin', 'walk', 'what', 'who', 'woman', 'year', 'yes']
Total classes: 32


In [ ]:
class_counts = {}

for cls in classes:
    cls_path = os.path.join(data_dir, cls)
    videos = os.listdir(cls_path)
    class_counts[cls] = len(videos)

for k,v in class_counts.items():
    print(f"{k}: {v}")

all: 19
before: 19
black: 19
book: 8
candy: 17
chair: 15
clothes: 12
computer: 20
cool: 19
cousin: 19
deaf: 19
drink: 18
fine: 18
finish: 19
go: 19
help: 17
hot: 20
like: 20
many: 20
mother: 18
no: 19
now: 20
orange: 19
table: 13
thanksgiving: 20
thin: 18
walk: 18
what: 19
who: 17
woman: 18
year: 18
yes: 19


In [ ]:
TARGET_VIDEOS = 20

In [ ]:
import cv2
import numpy as np
import mediapipe as mp

IMG_SIZE = 160   # reduced for speed
MAX_FRAMES = 20  # reduced for stability

mp_pose = mp.solutions.pose.Pose()
mp_hands = mp.solutions.hands.Hands(max_num_hands=2)

def sample_frames(frames, max_frames=20):
    if len(frames) < max_frames:
        while len(frames) < max_frames:
            frames.append(frames[-1])
    else:
        idx = np.linspace(0, len(frames)-1, max_frames).astype(int)
        frames = [frames[i] for i in idx]
    return frames

def extract_pose(frame):
    frame_rgb = cv2.cvtColor((frame*255).astype(np.uint8), cv2.COLOR_BGR2RGB)

    pose_features = []

    # Process pose landmarks
    results_pose = mp_pose.process(frame_rgb)
    if results_pose.pose_landmarks:
        for lm in results_pose.pose_landmarks.landmark:
            pose_features.extend([lm.x, lm.y, lm.z])
    else:
        pose_features.extend([0]*99) # 33 landmarks * 3 coordinates

    # Process hand landmarks
    results_hand = mp_hands.process(frame_rgb)
    hands_processed = 0
    if results_hand.multi_hand_landmarks:
        for hand_idx, hand in enumerate(results_hand.multi_hand_landmarks):
            if hand_idx >= 2: # Process a maximum of 2 hands
                break
            for lm in hand.landmark:
                pose_features.extend([lm.x, lm.y, lm.z])
            hands_processed += 1

    # Fill in zeros for any missing hands (up to 2 hands)
    for _ in range(2 - hands_processed):
        pose_features.extend([0]*63) # 21 landmarks * 3 coordinates

    return pose_features

In [ ]:
def augment_video(video):
    aug_video = []

    for frame in video:
        frame = frame.copy()

        # Flip
        if np.random.rand() < 0.5:
            frame = np.fliplr(frame)

        # Brightness
        frame = frame * (0.8 + 0.4*np.random.rand())

        # Rotation
        if np.random.rand() < 0.3:
            angle = np.random.randint(-15, 15)
            h, w = frame.shape[:2]
            M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1)
            frame = frame.astype(np.float32)
            frame = cv2.warpAffine(frame, M, (w, h))

        aug_video.append(frame)

    return np.array(aug_video)

In [ ]:
from tqdm import tqdm
import os
import numpy as np
import cv2

save_dir = "/content/drive/MyDrive/sanket2/processed_chunks"
os.makedirs(save_dir, exist_ok=True)

sample_idx = 0

for cls in tqdm(classes):
    cls_path = os.path.join(data_dir, cls)+
    videos = os.listdir(cls_path)

    label = gloss_to_id[cls]

    samples = []

    # -------- Process videos --------
    for vid in videos:
        video_path = os.path.join(cls_path, vid)

        cap = cv2.VideoCapture(video_path)
        frames = []

        count = 0
        while cap.isOpened() and count < MAX_FRAMES:
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
            frame = frame / 255.0

            frames.append(frame)
            count += 1

        cap.release()

        if len(frames) == 0:
            continue

        frames = sample_frames(frames, MAX_FRAMES)
        poses = [extract_pose(f) for f in frames]

        samples.append((np.array(frames, dtype=np.float16),
                        np.array(poses, dtype=np.float16)))

    # -------- Skip empty classes --------
    if not samples:
        continue

    # -------- BALANCING --------
    while len(samples) < TARGET_VIDEOS:
        samples.append(samples[np.random.randint(len(samples))])

    samples = samples[:TARGET_VIDEOS]

    # -------- SAVE IMMEDIATELY --------
    for frames, poses in samples:

        # Save original
        np.save(f"{save_dir}/rgb_{sample_idx}.npy", frames)
        np.save(f"{save_dir}/pose_{sample_idx}.npy", poses)
        np.save(f"{save_dir}/label_{sample_idx}.npy", label)

        sample_idx += 1

        # Augmentation for minority classes
        if class_counts[cls] < 18:
            aug = augment_video(frames)

            np.save(f"{save_dir}/rgb_{sample_idx}.npy", aug.astype(np.float16))
            np.save(f"{save_dir}/pose_{sample_idx}.npy", poses)
            np.save(f"{save_dir}/label_{sample_idx}.npy", label)

            sample_idx += 1

print("✅ Processing complete! Total samples:", sample_idx)

100%|██████████| 32/32 [06:47<00:00, 12.73s/it]

✅ Processing complete! Total samples: 780


In [ ]:
import os
from sklearn.model_selection import train_test_split
import numpy as np

data_path = "/content/drive/MyDrive/sanket2/processed_chunks"

# Get all sample indices
files = os.listdir(data_path)
indices = sorted(list(set([f.split("_")[1].split(".")[0] for f in files])))

indices = np.array(indices, dtype=int)

print("Total samples:", len(indices))

# Split
train_idx, temp_idx = train_test_split(indices, test_size=0.3, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

print("Train:", len(train_idx))
print("Val:", len(val_idx))
print("Test:", len(test_idx))

Total samples: 780
Train: 546
Val: 117
Test: 117


In [ ]:
split_path = "/content/drive/MyDrive/sanket2/splits"
os.makedirs(split_path, exist_ok=True)

np.save(split_path + "/train_idx.npy", train_idx)
np.save(split_path + "/val_idx.npy", val_idx)
np.save(split_path + "/test_idx.npy", test_idx)

print("✅ Splits saved!")

✅ Splits saved!


In [ ]:
import os

base_path = "/content/drive/MyDrive/sanket2"

def check_path(path, description):
    if os.path.exists(path):
        print(f"✅ {description} exists: {path}")
        return True
    else:
        print(f"❌ {description} NOT found: {path}")
        return False


print("\n🔍 Checking main folder...")
check_path(base_path, "Base folder (sanket2)")

# -----------------------------
# raw_videos
# -----------------------------
raw_videos_path = os.path.join(base_path, "raw_videos")

print("\n🔍 Checking raw_videos...")
if check_path(raw_videos_path, "raw_videos"):
    classes = os.listdir(raw_videos_path)
    print(f"📊 Found {len(classes)} class folders")
    print("Sample classes:", classes[:5])


# -----------------------------
# processed_chunks
# -----------------------------
processed_path = os.path.join(base_path, "processed_chunks")

print("\n🔍 Checking processed_chunks...")
if check_path(processed_path, "processed_chunks"):
    files = os.listdir(processed_path)

    rgb_files = [f for f in files if f.startswith("rgb_")]
    pose_files = [f for f in files if f.startswith("pose_")]
    label_files = [f for f in files if f.startswith("label_")]

    print(f"📦 RGB files: {len(rgb_files)}")
    print(f"📦 Pose files: {len(pose_files)}")
    print(f"📦 Label files: {len(label_files)}")

    if len(rgb_files) == len(pose_files) == len(label_files):
        print("✅ Processed data is consistent")
    else:
        print("⚠️ Mismatch in processed files!")


# -----------------------------
# splits
# -----------------------------
splits_path = os.path.join(base_path, "splits")

print("\n🔍 Checking splits...")
if check_path(splits_path, "splits"):
    train_file = os.path.join(splits_path, "train_idx.npy")
    val_file = os.path.join(splits_path, "val_idx.npy")
    test_file = os.path.join(splits_path, "test_idx.npy")

    train_ok = check_path(train_file, "train_idx.npy")
    val_ok = check_path(val_file, "val_idx.npy")
    test_ok = check_path(test_file, "test_idx.npy")

    if train_ok and val_ok and test_ok:
        print("✅ All split files exist")

        import numpy as np
        train_idx = np.load(train_file)
        val_idx = np.load(val_file)
        test_idx = np.load(test_file)

        print(f"📊 Train samples: {len(train_idx)}")
        print(f"📊 Val samples: {len(val_idx)}")
        print(f"📊 Test samples: {len(test_idx)}")


🔍 Checking main folder...
✅ Base folder (sanket2) exists: /content/drive/MyDrive/sanket2

🔍 Checking raw_videos...
✅ raw_videos exists: /content/drive/MyDrive/sanket2/raw_videos
📊 Found 32 class folders
Sample classes: ['book', 'drink', 'computer', 'before', 'chair']

🔍 Checking processed_chunks...
✅ processed_chunks exists: /content/drive/MyDrive/sanket2/processed_chunks
📦 RGB files: 780
📦 Pose files: 780
📦 Label files: 780
✅ Processed data is consistent

🔍 Checking splits...
✅ splits exists: /content/drive/MyDrive/sanket2/splits
✅ train_idx.npy exists: /content/drive/MyDrive/sanket2/splits/train_idx.npy
✅ val_idx.npy exists: /content/drive/MyDrive/sanket2/splits/val_idx.npy
✅ test_idx.npy exists: /content/drive/MyDrive/sanket2/splits/test_idx.npy
✅ All split files exist
📊 Train samples: 546
📊 Val samples: 117
📊 Test samples: 117
